In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('imports OK')

---
## Problem 5: LTV Optimization via Offline Reinforcement Learning

### Business Context
Standard recommenders optimize immediate CTR. This misses long-term value — a user who clicks but churns is worse than one who engages moderately but stays for years. Optimize 12-month Customer Lifetime Value (CLV) using offline RL.

### Why Offline RL?
- **Online RL** (A/B test every policy) is too risky and slow for production
- **Offline RL** learns from logged historical interactions
- **Key challenge**: Q-values for out-of-distribution (OOD) actions are overestimated → model confidently recommends actions never tried before

### MDP Formulation
```
State  s_t:  customer history, profile, behavioral signals, product holdings
Action a_t:  recommendation (item from catalog)
Reward r_t:  revenue - costs + retention_signal
Discount γ:  0.99 (monthly steps)
Objective:   max_π E_π[Σ γ^t r_t]
```

### Conservative Q-Learning (CQL)
Standard Q-learning overestimates unseen actions. CQL adds a penalty:
```
L_CQL = L_Bellman + α * (log Σ_a' exp(Q(s,a')) - E_{a~D}[Q(s,a)])
```
- First term: push Q-values DOWN for all actions (conservative)
- Second term: push Q-values UP for in-distribution actions (preserve accuracy)
- Result: conservative for OOD actions, accurate for seen actions

### Doubly Robust Off-Policy Evaluation (DR-OPE)
Before deploying, estimate the value of the new policy using logged data:
```
V_DR = V_DM + E[(π_e/π_b) * (r - Q_DM(s,a))]
```
- **DM** (Direct Method): use Q-network to estimate value. Low variance, potentially biased.
- **IS** (Importance Sampling): reweight by policy ratio. Unbiased, high variance.
- **DR**: combines both. Correct if either DM or IS is correct — doubly robust.

In [ ]:
# --- CQL Loss and DR-OPE ---

class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )
    def forward(self, state):
        return self.net(state)   # (B, n_actions)

def cql_loss(q_net, target_net, states, actions, rewards, next_states,
             dones, gamma=0.99, alpha=1.0):
    """
    Conservative Q-Learning loss.
    alpha: CQL regularization strength
    """
    B = states.shape[0]

    # --- Standard Bellman (TD) loss ---
    with torch.no_grad():
        next_q = target_net(next_states).max(dim=1).values    # (B,)
        targets = rewards + gamma * next_q * (1 - dones)      # (B,)

    q_vals = q_net(states)                                    # (B, n_actions)
    q_sa   = q_vals.gather(1, actions.unsqueeze(1)).squeeze() # Q(s,a) for taken actions
    td_loss = F.mse_loss(q_sa, targets)

    # --- CQL penalty ---
    # Push all Q-values down (log-sum-exp term), push in-distribution Q up
    logsumexp_q   = torch.logsumexp(q_vals, dim=1).mean()     # all actions
    in_data_q     = q_sa.mean()                               # logged actions only
    cql_penalty   = logsumexp_q - in_data_q

    total_loss = td_loss + alpha * cql_penalty
    return total_loss, td_loss.item(), cql_penalty.item()

# --- Doubly Robust OPE ---
def doubly_robust_ope(q_net, states, actions, rewards, behavior_probs, eval_probs,
                      max_weight=20.0):
    """
    Estimate value of evaluation policy using logged data.
    behavior_probs: π_b(a|s) — probability of logged action under behavior policy
    eval_probs:     π_e(a|s) — probability under evaluation policy
    """
    with torch.no_grad():
        q_vals = q_net(states)                                  # (B, n_actions)
        q_sa   = q_vals.gather(1, actions.unsqueeze(1)).squeeze()  # Q(s,a)

        # Direct Method: expected value under eval policy
        eval_probs_all = torch.softmax(torch.randn_like(q_vals), dim=1)  # placeholder
        v_dm = (q_vals * eval_probs_all).sum(dim=1).mean()

        # Importance weights (clipped)
        is_weights = (eval_probs / behavior_probs.clamp(min=1e-6)).clamp(max=max_weight)

        # DR correction
        correction = (is_weights * (rewards - q_sa)).mean()
        v_dr = v_dm + correction

    return {'V_DM': v_dm.item(), 'V_DR': v_dr.item()}

# Simulation
state_dim, n_actions, B = 16, 50, 128
q_net     = QNetwork(state_dim, n_actions)
target_net = QNetwork(state_dim, n_actions)
target_net.load_state_dict(q_net.state_dict())
optimizer = torch.optim.Adam(q_net.parameters(), lr=3e-4)

states      = torch.randn(B, state_dim)
actions     = torch.randint(0, n_actions, (B,))
rewards     = torch.randn(B)
next_states = torch.randn(B, state_dim)
dones       = torch.zeros(B)

loss, td, cql_pen = cql_loss(q_net, target_net, states, actions, rewards, next_states, dones)
print(f'CQL Loss: {loss.item():.4f}  (TD: {td:.4f}, CQL penalty: {cql_pen:.4f})')

behavior_probs = torch.rand(B).clamp(0.05, 0.3)
eval_probs     = torch.rand(B).clamp(0.05, 0.3)
ope = doubly_robust_ope(q_net, states, actions, rewards, behavior_probs, eval_probs)
print(f'OPE Estimates: DM={ope["V_DM"]:.4f}  DR={ope["V_DR"]:.4f}')

### L6 Interview Q&A

**Q: What is the fundamental challenge of offline RL vs supervised learning?**  
A: In supervised learning, the training distribution matches the test distribution. In offline RL, the learned policy may select actions that were never (or rarely) taken in the logged data. Q-values for these OOD actions are extrapolated and typically overestimated, leading the agent to confidently recommend actions it has never seen work. CQL penalizes high Q-values for OOD actions, keeping the policy within the support of the logged data.

**Q: CTR drops 8% but CLV increases 3% in the A/B test. How do you explain this to stakeholders?**  
A: This is the expected trade-off. The RL policy sacrifices immediate engagement for long-term value — it recommends items that are slightly less click-worthy but drive deeper engagement, repeat purchases, or retention. Show the unit economics: 3% CLV improvement on a customer base worth $100M annually = $3M incremental revenue. The 8% CTR drop is the cost we pay for that.

**Q: How would you validate the new policy before deploying it?**  
A: DR-OPE on the holdout logged data — if the estimated value improvement is ≥3% with a bootstrapped 95% CI that doesn't include 0, proceed to a small canary (5% traffic). Run canary for 2-4 weeks (long enough to see downstream LTV signal). Gate on: OPE estimate, KL divergence of action distribution from behavior policy ≤ 2 nats, in-distribution rate ≥ 0.85.

### Common Pitfalls
- Using current bid/recommendation as the label (conflates historical policy with optimal)
- Not clipping IS weights — extreme weights (>20) cause OPE variance to explode
- Deploying without OPE and deployment gates — high risk of silent regression